# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os
from dotenv import load_dotenv

from datetime import datetime
from typing import List, Dict

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import BaseMessage
from lib.tooling import tool
from lib.evaluation import EvaluationReport
from lib.parsers import PydanticOutputParser

import chromadb

from tavily import TavilyClient


In [3]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
@tool
def retrieve_game(query: str, n_results: int = 3) -> List[Dict]:
    """
    Retrieve games from the vector DB

    Args:
        query (str): a question about game industry
        n_results (int): Maximum number of results to return per query (default: 3)
        
    Returns:
        List[Dict]: 

    """

    try:
        chroma_client = chromadb.PersistentClient(path="chromadb")
        collection = chroma_client.get_collection("udaplay")
        
        if not collection:
            raise ValueError("Failed to find the chromadb collection")
        
        query_results = collection.query(
            query_texts=query,
            n_results=n_results,
            include=["metadatas"]
        )

        results: List[Dict] = []
        for query_result in query_results["metadatas"][0]:
            result = {
                "Platform": query_result["Platform"],
                "Name": query_result["Name"],
                "YearOfRelease": query_result["YearOfRelease"],
                "Description": query_result["Description"]
            }
            results.append(result)

        return results
    
    except Exception as e:
        print(f"Error retrieving games: {e}")
        return []


#### Evaluate Retrieval Tool

In [5]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: List[str]) -> Dict:
    """
    Evaluate retrieved docs for relevance to question

    Args:
        question (str): Original question from user
        retrieved_docs (List[str]): Retrieved documents most similar to the user query in the Vector Database

    Returns:
        Dict:
    """

    evalutation_prompt = f"""
        Your task is to evaluate if the documents are enough to respond the query.
        Give a detailed explanation, so it's possible to take an action to accept it or not.
        
        User Query: {question}
        Retrieved Documents: {retrieved_docs}
    """
    
    llm_evaluator = LLM(model="gpt-4o-mini")

    evaluator_response = llm_evaluator.invoke(
        input=evalutation_prompt,
        response_format=EvaluationReport
    )

    parser = PydanticOutputParser(model_class=EvaluationReport)
    try:
        return parser.parse(evaluator_response)
    except Exception as e:
        print(f"Debug: Structured parsing error: {e}")
        print(f"Debug: Evaluator response content: {evaluator_response.content}")
        return EvaluationReport(
            useful=False,
            description=f"Fallback evaluation due to parsing error: {str(e)}"
        )

#### Game Web Search Tool

In [6]:
@tool
def game_web_search(question: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API
    
    Args:
        question (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    
    Returns:
        Dict:
    """

    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient()

    search_result = client.search(
        query=question,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )

    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question
        }
    }

    return formatted_results

### Agent

In [7]:
tools = [retrieve_game, evaluate_retrieval, game_web_search]

uda_play_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are a researcher at a gaming anlytics company.\n"
        "Answer user questions about games based on the documents from the internal retrieval system "
        "as well as those fetched by using the given web search tool.\n\n"
        "Upon a user question:\n"
        "Step #1: Query the relevant documents from the local retrieval system.\n"
        "Step #2: Evaluate the retrieved documents by using the given evaluation tool.\n"
        "Step #3: Based on the evaluation result, determine whether you can answer the question truthfully.\n"
        "Step #4: If not, gather more information about the question by using the given web search tool.\n"
        "Step #5: Build the answer based on the information gatehered by the previous steps, exactly followng this format:\n"
        "TITLE: Game title\n"
        "GENRES: Game genres\n"
        "PLATFORMS: Platforms\n"
        "PUBLISHER: Publisher information\n"
        "RELEASE YEAR: Year of release\n"
        "DESCIRPTION: Game description\n"
        "CONFIDENCE LEVEL: Report the confidence level in answer\n"
        "SOURCES: Cite information sources\n"
    ),
    tools=tools,
)

In [8]:
session_id = "game_research_001"
queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?",
    "Who developed FIFA 21?"
]
runs = []

for index, query in enumerate(queries):
    print(f"Session: {session_id}, Query {index}: {query}")
    run = uda_play_agent.invoke(query=query, session_id=session_id)
    runs.append(run)


Session: game_research_001, Query 0: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Session: game_research_001, Query 1: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Session: game_research_001, Query 2: Was Mortal

In [9]:
def print_messages(messages: List[BaseMessage]):
    for m in messages: 
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

def print_final_answer(query: str, messages: List[BaseMessage]):
    print("="*50)
    print("User Question:")
    print(f"{query}")
    print("\nAgent's Answer:")
    print(messages[-1].content)
    print("="*50)


In [10]:
print(f"Messages from run #1:")
messages = runs[0].get_final_state()["messages"]
    
print_messages(messages)
print("\n\n")
print_final_answer(queries[0], messages)

Messages from run #1:
 -> (role = system, content = You are a researcher at a gaming anlytics company.
Answer user questions about games based on the documents from the internal retrieval system as well as those fetched by using the given web search tool.

Upon a user question:
Step #1: Query the relevant documents from the local retrieval system.
Step #2: Evaluate the retrieved documents by using the given evaluation tool.
Step #3: Based on the evaluation result, determine whether you can answer the question truthfully.
Step #4: If not, gather more information about the question by using the given web search tool.
Step #5: Build the answer based on the information gatehered by the previous steps, exactly followng this format:
TITLE: Game title
GENRES: Game genres
PLATFORMS: Platforms
PUBLISHER: Publisher information
RELEASE YEAR: Year of release
DESCIRPTION: Game description
CONFIDENCE LEVEL: Report the confidence level in answer
SOURCES: Cite information sources
, tool_calls = None)


In [11]:
print(f"Messages from run #2:")
messages = runs[1].get_final_state()["messages"]
    
print_messages(messages)
print("\n\n")
print_final_answer(queries[1], messages)

Messages from run #2:
 -> (role = system, content = You are a researcher at a gaming anlytics company.
Answer user questions about games based on the documents from the internal retrieval system as well as those fetched by using the given web search tool.

Upon a user question:
Step #1: Query the relevant documents from the local retrieval system.
Step #2: Evaluate the retrieved documents by using the given evaluation tool.
Step #3: Based on the evaluation result, determine whether you can answer the question truthfully.
Step #4: If not, gather more information about the question by using the given web search tool.
Step #5: Build the answer based on the information gatehered by the previous steps, exactly followng this format:
TITLE: Game title
GENRES: Game genres
PLATFORMS: Platforms
PUBLISHER: Publisher information
RELEASE YEAR: Year of release
DESCIRPTION: Game description
CONFIDENCE LEVEL: Report the confidence level in answer
SOURCES: Cite information sources
, tool_calls = None)


In [12]:
print(f"Messages from run #3:")
messages = runs[2].get_final_state()["messages"]
    
print_messages(messages)
print("\n\n")
print_final_answer(queries[2], messages)

Messages from run #3:
 -> (role = system, content = You are a researcher at a gaming anlytics company.
Answer user questions about games based on the documents from the internal retrieval system as well as those fetched by using the given web search tool.

Upon a user question:
Step #1: Query the relevant documents from the local retrieval system.
Step #2: Evaluate the retrieved documents by using the given evaluation tool.
Step #3: Based on the evaluation result, determine whether you can answer the question truthfully.
Step #4: If not, gather more information about the question by using the given web search tool.
Step #5: Build the answer based on the information gatehered by the previous steps, exactly followng this format:
TITLE: Game title
GENRES: Game genres
PLATFORMS: Platforms
PUBLISHER: Publisher information
RELEASE YEAR: Year of release
DESCIRPTION: Game description
CONFIDENCE LEVEL: Report the confidence level in answer
SOURCES: Cite information sources
, tool_calls = None)


In [13]:
print(f"Messages from run #4:")
messages = runs[3].get_final_state()["messages"]
    
print_messages(messages)
print("\n\n")
print_final_answer(queries[3], messages)

Messages from run #4:
 -> (role = system, content = You are a researcher at a gaming anlytics company.
Answer user questions about games based on the documents from the internal retrieval system as well as those fetched by using the given web search tool.

Upon a user question:
Step #1: Query the relevant documents from the local retrieval system.
Step #2: Evaluate the retrieved documents by using the given evaluation tool.
Step #3: Based on the evaluation result, determine whether you can answer the question truthfully.
Step #4: If not, gather more information about the question by using the given web search tool.
Step #5: Build the answer based on the information gatehered by the previous steps, exactly followng this format:
TITLE: Game title
GENRES: Game genres
PLATFORMS: Platforms
PUBLISHER: Publisher information
RELEASE YEAR: Year of release
DESCIRPTION: Game description
CONFIDENCE LEVEL: Report the confidence level in answer
SOURCES: Cite information sources
, tool_calls = None)


### (Optional) Advanced

In [14]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes